In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

STUDENT_ID = "TUPM-26-4014"
SEED_DIGIT = int(STUDENT_ID[-1])  # 4
NUM_FEATURES = 5 + (SEED_DIGIT % 3)  # 6

# Ensure output directory exists
Path("output").mkdir(exist_ok=True)

# 1. Load SECOM dataset safely (checking local vs data/ folder paths)
possible_paths = [
    (Path("data/secom.data"), Path("data/secom_labels.data")),
    (Path("secom.data"), Path("secom_labels.data")),
    (Path("../data/secom.data"), Path("../data/secom_labels.data")),
]

data_path, labels_path = None, None
for d_p, l_p in possible_paths:
  if d_p.exists() and l_p.exists():
    data_path, labels_path = d_p, l_p
    break

if data_path is None:
  raise FileNotFoundError(
      "Could not find 'secom.data' and 'secom_labels.data'. Please ensure"
      " they are in your working directory or a 'data/' folder."
  )

# Read whitespace-delimited files
X = pd.read_csv(data_path, sep=r"\s+", header=None)
y = pd.read_csv(labels_path, sep=r"\s+", header=None)

# Combine into a single DataFrame for uniform handling
df = X.copy()
df["Outcome"] = y.iloc[:, 0]

# Identify features with missing or constant observed values
features = df.iloc[:, :-1]  # exclude outcome label
missing_vals = features.isnull().mean()
constant_mask = features.nunique() <= 1
valid_features = features.loc[:, ~constant_mask & (missing_vals < 0.5)]

# 2. Reproducibly select features using random sampling
np.random.seed(SEED_DIGIT)
selected_feature_indices = np.random.choice(
    valid_features.columns, size=NUM_FEATURES, replace=False
)
selected_df = valid_features[selected_feature_indices]

# 3. Calculate feature-wise means, medians, std deviations, and missing percentages
means = selected_df.mean()
medians = selected_df.median()
stds = selected_df.std()
missing_pcts = selected_df.isnull().mean() * 100

# 4. Compute robust standardized values using median and MAD
mad = (selected_df - medians).abs().median()
mad_safe = mad.replace(0, 1.0)  # Handle zero MAD safely
robust_z = 0.6745 * (selected_df - medians) / mad_safe

# 5. Identify measurements with absolute robust standardized values > 3.5
flagged_obs = robust_z.abs() > 3.5
total_flagged = int(flagged_obs.sum().sum())

# 6. Rank features by percentage of flagged observations
flagged_pct_per_feature = flagged_obs.mean() * 100
ranked_features = flagged_pct_per_feature.sort_values(ascending=False)

highest_missing_feat = missing_pcts.idxmax()
highest_missing_val = missing_pcts.max()
highest_flagged_feat = flagged_pct_per_feature.idxmax()

# Export summary CSV
summary_df = pd.DataFrame({
    "Feature": ranked_features.index,
    "Missing_Pct": missing_pcts[ranked_features.index],
    "Flagged_Pct": ranked_features.values,
})
summary_df.to_csv("output/measurement_quality.csv", index=False)

print("--- Exercise 1 Assessment Data ---")
print(f"Selected Features: {list(selected_feature_indices)}")
print(
    f"Highest Missing Percentage: Feature {highest_missing_feat}"
    f" ({highest_missing_val:.2f}%)"
)
print(f"Feature with Highest Flagged Percentage: {highest_flagged_feat}")
print(f"Total Flagged Observations: {total_flagged}")
print("Vectorized Results Verified: True")

--- Exercise 1 Assessment Data ---
Selected Features: [441, 219, 89, 548, 587, 137]
Highest Missing Percentage: Feature 548 (16.59%)
Feature with Highest Flagged Percentage: 548
Total Flagged Observations: 312
Vectorized Results Verified: True
